# HMS – Harmful Brain Activity Classification: a simple scikit-learn baseline

**Task.** Each sample is an EEG segment from a critically ill patient. Several experts voted on which of six
patterns it shows: seizure, LPD, GPD, LRDA, GRDA or other. The target is the *distribution* of votes,
and submissions are scored with the KL divergence between the true and the predicted distribution (lower is better).

**Approach.**
1. **Labels**: for each EEG, turn the expert votes into a probability distribution over the six classes.
2. **Features** (88 per EEG): from the official Kaggle spectrograms we take the 10 minutes around the labelled
   segment and compute log band power (delta, theta, alpha, beta) for each of the four scalp regions, summarised
   over time (mean, std, min, max), plus relative band power and left–right asymmetry.
3. **Models**: `LogisticRegression` and `HistGradientBoostingClassifier` from scikit-learn.
   Soft labels are handled by repeating each EEG once per class with `sample_weight` = vote fraction;
   the weighted cross-entropy is then equal to the KL divergence up to a constant.
4. **Validation**: 5-fold `GroupKFold` grouped by `patient_id`, so no patient appears in both the training and
   the validation part of a fold. The model with the better cross-validation (CV) score is used for the submission.
5. **Submission**: refit the chosen model on all training data and predict the hidden test set.

**Settings** (next cell): `MODE = "full"` uses all training spectrograms (about 10 minutes on a Kaggle CPU);
`MODE = "quick"` uses a random 3,000 for a fast check. `RUN_CV = False` skips cross-validation.

Runs on CPU only; internet must be off for Kaggle submissions.

In [1]:
import os
import time
import warnings
from glob import glob

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Some spectrograms contain frequency bands that are entirely NaN; numpy warns about the
# resulting empty means. These cases are handled explicitly below, so the warnings are muted.
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------------------------------------------------------- settings
MODE = "quick"      # "full": all training spectrograms | "quick": a random subset for a fast check
RUN_CV = True      # True: run 5-fold cross-validation, report CV scores and pick the better model
MAX_SPECS = {"full": None, "quick": 3000}[MODE]   # number of training spectrograms to read (None = all)
N_FOLDS = 5        # number of cross-validation folds
N_JOBS = -1        # number of CPU cores used to read spectrograms in parallel (-1 = all cores)
SEED = 42          # random seed for subsampling and for the gradient-boosting model

# ----------------------------------------------------------------------------- data location
# Kaggle mounts the competition data at different paths depending on the notebook environment.
# HMS_DATA can be set to run the notebook elsewhere (e.g. on a local copy of the data).
CANDIDATES = ["/kaggle/input/competitions/hms-harmful-brain-activity-classification",
              "/kaggle/input/hms-harmful-brain-activity-classification"]
DATA = os.environ.get("HMS_DATA") or next((p for p in CANDIDATES if os.path.exists(f"{p}/train.csv")), None)
if DATA is None:   # last resort: search the input folder (slow, about 1-2 minutes)
    DATA = os.path.dirname(sorted(glob("/kaggle/input/**/train.csv", recursive=True), key=len)[0])
print("DATA =", DATA)

# ----------------------------------------------------------------------------- constants
# Vote columns in train.csv, in the order required by the submission file.
CLASSES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
# The official spectrograms average the bipolar EEG channels into four scalp regions:
# LL = left lateral, RL = right lateral, LP = left parasagittal, RP = right parasagittal.
REGIONS = ["LL", "RL", "LP", "RP"]
# Classical EEG frequency bands in Hz (the spectrograms cover roughly 0.5-20 Hz).
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 20)}

DATA = /kaggle/input/competitions/hms-harmful-brain-activity-classification


## 1. Labels: one row per EEG, votes normalised to a probability distribution

`train.csv` has several rows (labelled sub-segments) per `eeg_id`. We sum the votes over these rows,
divide by the total number of votes, and keep the range of label offsets to locate the window in the spectrogram.

In [2]:
train = pd.read_csv(f"{DATA}/train.csv")

# One row per EEG: spectrogram id, first/last label offset (seconds), patient id and summed votes per class.
agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
agg.update({c: "sum" for c in CLASSES})
df = train.groupby("eeg_id").agg(agg)
df.columns = ["spectrogram_id", "off_min", "off_max", "patient_id"] + CLASSES

# Convert vote counts into a probability distribution (each row sums to 1).
df["num_votes"] = df[CLASSES].sum(axis=1)
df[CLASSES] = df[CLASSES].values / df["num_votes"].values[:, None]
df = df.reset_index()

# Quick mode: keep only the EEGs of a random subset of spectrograms.
if MAX_SPECS:
    keep = pd.Series(df.spectrogram_id.unique()).sample(n=min(MAX_SPECS, df.spectrogram_id.nunique()),
                                                         random_state=SEED)
    df = df[df.spectrogram_id.isin(keep)].reset_index(drop=True)

print(df.shape, "EEGs from", df.patient_id.nunique(), "patients,", df.spectrogram_id.nunique(), "spectrograms")
df.head()

(4738, 12) EEGs from 1163 patients, 3000 spectrograms


,eeg_id,spectrogram_id,off_min,off_max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,num_votes
0,568657,789577333,0.0,16.0,20654,0.0,0.0,0.250000,0.000000,0.166667,0.583333,48
1,751790,618728447,908.0,908.0,38549,0.0,0.0,1.000000,0.000000,0.000000,0.000000,1
2,2078097,2074135650,3342.0,3342.0,61174,0.0,0.0,0.000000,0.000000,0.000000,1.000000,2
3,2482631,978166025,1902.0,1944.0,20606,0.0,0.0,0.133333,0.066667,0.133333,0.666667,105
4,2918824,1211648246,3462.0,3462.0,14965,1.0,0.0,0.000000,0.000000,0.000000,0.000000,2


## 2. Features from the official spectrograms

Each spectrogram file has one row every 2 seconds and 400 power columns named `<region>_<frequency>`
(e.g. `LL_1.17`). We take a 10-minute window (300 rows) and reduce it to 88 numbers per EEG:

| Feature group | Count |
|---|---|
| Log band power per region and band: mean, std, min, max over time | 4 regions × 4 bands × 4 = 64 |
| Relative band power (band power / total power of the region) | 4 × 4 = 16 |
| Left–right asymmetry of mean log band power (lateral and parasagittal chains) | 2 × 4 = 8 |

Spectrogram files are read in parallel because reading them is the slowest step of the notebook.

In [3]:
def band_columns(columns):
    """Return {(region, band): [column indices]} for the 400 spectrogram columns like 'LL_1.17'."""
    out = {}
    for r in REGIONS:
        for b, (lo, hi) in BANDS.items():
            out[(r, b)] = [i for i, c in enumerate(columns)
                           if c.startswith(r + "_") and lo <= float(c.split("_")[1]) < hi]
    return out


def window_features(spec, cols):
    """Compute the 88 features for one 10-minute window.

    spec: float array of shape (time steps, 400 power columns).
    cols: output of band_columns().
    """
    feats = {}
    total = {r: 0.0 for r in REGIONS}   # total power per region, for relative band power
    band_power = {}
    for (r, b), idx in cols.items():
        p = np.nanmean(spec[:, idx], axis=1)                     # mean power in the band, per time step
        lp = np.log1p(np.nan_to_num(p, nan=0.0).clip(min=0))     # log scale; missing values -> 0
        feats[f"{r}_{b}_mean"] = lp.mean()
        feats[f"{r}_{b}_std"] = lp.std()
        feats[f"{r}_{b}_min"] = lp.min()
        feats[f"{r}_{b}_max"] = lp.max()
        band_power[(r, b)] = np.nanmean(p) if np.isfinite(p).any() else 0.0
        total[r] += band_power[(r, b)]
    for (r, b), v in band_power.items():                         # share of the region's total power
        feats[f"{r}_{b}_rel"] = v / (total[r] + 1e-6)
    for b in BANDS:                                              # left minus right hemisphere
        feats[f"asym_lat_{b}"] = feats[f"LL_{b}_mean"] - feats[f"RL_{b}_mean"]
        feats[f"asym_par_{b}"] = feats[f"LP_{b}_mean"] - feats[f"RP_{b}_mean"]
    return feats


def spectrogram_features(path, windows, is_test):
    """Read one spectrogram file and compute features for every EEG that uses it.

    windows: list of (eeg_id, off_min, off_max); offsets are the label positions in seconds.
    """
    spec = pd.read_parquet(path)
    cols = band_columns(list(spec.columns[1:]))
    spec = spec.iloc[:, 1:].to_numpy(dtype=np.float32)          # drop the 'time' column
    rows = []
    for eeg_id, off_min, off_max in windows:
        if is_test:
            # Test spectrograms are short and centred on the segment: take their middle 10 minutes.
            start = max(0, (len(spec) - 300) // 2)
        else:
            # Training spectrograms cover the whole recording. Start at the midpoint of the labelled
            # offsets; offsets are in seconds and rows are 2 s apart, hence the division by 4 (= /2 /2).
            start = int((off_min + off_max) // 4)
        feats = window_features(spec[start:start + 300], cols)
        feats["eeg_id"] = eeg_id
        rows.append(feats)
    return rows


def build_features(meta, spec_dir, is_test=False):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(f"{spec_dir}/{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


X_df = build_features(df, f"{DATA}/train_spectrograms")
df = df.merge(X_df, on="eeg_id")
FEATS = [c for c in X_df.columns if c != "eeg_id"]
print(len(FEATS), "features")

/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_16/2784577015.py:21: RuntimeWarning: Mean of empt

3000 spectrograms -> 4738 feature rows in 54s
88 features


## 3. Soft-label training and patient-grouped cross-validation

**Soft labels in scikit-learn.** scikit-learn classifiers expect one class per sample. We therefore repeat each EEG
six times, once per class, and weight each copy by that class's vote fraction. Minimising the weighted
cross-entropy is then the same as minimising the KL divergence to the vote distribution.

**Cross-validation (CV).** The training data are split into 5 folds *by patient*. Each model is trained on 4 folds
and evaluated on the held-out fold; the out-of-fold predictions give an estimate of performance on unseen patients.
Grouping by patient matters: EEGs of the same patient are very similar, and a random split would overestimate
performance. We report the CV score on all EEGs and on EEGs with at least 10 votes (more reliable labels),
next to the score of a trivial model that always predicts the average vote distribution (the *prior*).

In [4]:
def kl_score(y_true, y_pred, eps=1e-15):
    """Mean KL divergence KL(y_true || y_pred) over samples, as in the competition metric."""
    p = np.clip(y_pred, eps, 1)
    p = p / p.sum(axis=1, keepdims=True)
    t = y_true
    return np.mean(np.sum(np.where(t > 0, t * np.log(np.where(t > 0, t, 1) / p), 0), axis=1))


def expand(X, Y):
    """Turn soft labels into weighted hard labels.

    Each sample is repeated once per class (label = class index, weight = vote fraction);
    copies with zero weight are dropped.
    """
    n, k = Y.shape
    Xr = np.repeat(X, k, axis=0)
    yr = np.tile(np.arange(k), n)
    wr = Y.reshape(-1)
    keep = wr > 0
    return Xr[keep], yr[keep], wr[keep]


# The two candidate models. Logistic regression needs standardised features; gradient boosting does not.
MODELS = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)),
    "hgb": lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                                  max_leaf_nodes=31, l2_regularization=1.0,
                                                  random_state=SEED),
}


def fit(name, X, Y):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = MODELS[name]()
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


X = np.nan_to_num(df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
Y = df[CLASSES].to_numpy()
groups = df["patient_id"].to_numpy()
prior = np.tile(Y.mean(axis=0), (len(Y), 1))   # trivial reference: always predict the mean distribution
hi = df["num_votes"].to_numpy() >= 10          # EEGs with at least 10 expert votes

if RUN_CV:
    oof = {name: np.zeros_like(Y) for name in MODELS}   # out-of-fold predictions per model
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(X, Y, groups)):
        for name in MODELS:
            oof[name][va] = fit(name, X[tr], Y[tr]).predict_proba(X[va])
        print(f"fold {fold}: " + ", ".join(f"{n}={kl_score(Y[va], oof[n][va]):.4f}" for n in MODELS))
    print(f"\nCV KL (all EEGs)        prior={kl_score(Y, prior):.4f} " +
          " ".join(f"{n}={kl_score(Y, oof[n]):.4f}" for n in MODELS))
    print(f"CV KL (>=10 votes only) prior={kl_score(Y[hi], prior[hi]):.4f} " +
          " ".join(f"{n}={kl_score(Y[hi], oof[n][hi]):.4f}" for n in MODELS))
    BEST = min(MODELS, key=lambda n: kl_score(Y, oof[n]))   # lower KL is better
else:
    BEST = "hgb"   # default when CV is skipped
print("model for submission:", BEST)

fold 0: logreg=1.1914, hgb=1.0574
fold 1: logreg=1.1183, hgb=1.0798
fold 2: logreg=1.1207, hgb=1.1474
fold 3: logreg=1.2108, hgb=1.1232
fold 4: logreg=1.1222, hgb=0.9671

CV KL (all EEGs)        prior=1.2450 logreg=1.1527 hgb=1.0750
CV KL (>=10 votes only) prior=1.1824 logreg=1.1133 hgb=1.0783
model for submission: hgb


## 4. Fit the chosen model on all training data and predict the test set

The visible `test.csv` contains a single example. When the notebook is submitted, Kaggle re-runs it
with the hidden test set in its place, so this cell must work for any number of test EEGs.

In [5]:
final = fit(BEST, X, Y)

test = pd.read_csv(f"{DATA}/test.csv")
T_df = build_features(test, f"{DATA}/test_spectrograms", is_test=True)
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")   # keep the order of test.csv
XT = np.nan_to_num(T_df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

# Submission format: eeg_id plus one predicted probability per class (each row sums to 1).
sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[CLASSES] = final.predict_proba(XT)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
(1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.289826,0.03097,0.002351,0.189841,0.0349,0.452111
